In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_3bac2cb77dd2dbe0f97817b48726bb2f"

In [ ]:
!kaggle datasets download -d shayanfazeli/heartbeat
!unzip -q heartbeat.zip

Dataset URL: https://www.kaggle.com/datasets/shayanfazeli/heartbeat
License(s): unknown
100% 98.8M/98.8M [00:01<00:00, 89.4MB/s]



In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Load MIT-BIH train and test sets
train_df = pd.read_csv("/content/mitbih_train.csv", header=None)
test_df = pd.read_csv("/content/mitbih_test.csv", header=None)

# 2. Separate features (X) and labels (y)
X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values

X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values

print("classes:", np.unique(y_train))

class_counts = pd.Series(y_train).value_counts()
print(class_counts)
# 3. Filter for ONLY the majority class (Class 0) for training
X_train_major = X_train[y_train == 0]

print(f"Full Train shape: {X_train.shape}")
print(f"Majority Class Train shape: {X_train_major.shape}")
print(f"Test shape: {X_test.shape}")

classes: [0. 1. 2. 3. 4.]
0.0    72471
4.0     6431
2.0     5788
1.0     2223
3.0      641
Name: count, dtype: int64
Full Train shape: (87554, 187)
Majority Class Train shape: (72471, 187)
Test shape: (21892, 187)


In [ ]:
# 1D-CNN Autoencoder Architecture
input_sig = layers.Input(shape=(187, 1))

# Encoder
x = layers.Conv1D(32, kernel_size=5, activation="relu", padding="same")(input_sig)
x = layers.MaxPooling1D(2, padding="same")(x)
x = layers.Conv1D(16, kernel_size=5, activation="relu", padding="same")(x)
encoded = layers.MaxPooling1D(2, padding="same")(x)

# Decoder
x = layers.Conv1D(16, kernel_size=5, activation="relu", padding="same")(encoded)
x = layers.UpSampling1D(2)(x)
x = layers.Conv1D(32, kernel_size=5, activation="relu", padding="same")(x)
x = layers.UpSampling1D(2)(x)
# Crop to match original 187 length
x = layers.Cropping1D(cropping=(0, 1))(x)
decoded = layers.Conv1D(1, kernel_size=5, activation="linear", padding="same")(x)

conv_autoencoder = keras.Model(input_sig, decoded)
conv_autoencoder.compile(optimizer="adam", loss="mse")

In [ ]:
history = conv_autoencoder.fit(
    X_train_major, X_train_major,
    epochs=30,
    batch_size=128,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 32s 70ms/step - loss: 6.3571e-04 - val_loss: 3.5991e-04
Epoch 2/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 43s 75ms/step - loss: 2.8903e-04 - val_loss: 2.4257e-04
Epoch 3/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 42s 93ms/step - loss: 2.1316e-04 - val_loss: 2.1278e-04
Epoch 4/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 41s 90ms/step - loss: 1.7967e-04 - val_loss: 1.6022e-04
Epoch 5/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 32s 70ms/step - loss: 1.5817e-04 - val_loss: 1.4730e-04
Epoch 6/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 33s 73ms/step - loss: 1.4799e-04 - val_loss: 1.3091e-04
Epoch 7/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 32s 71ms/step - loss: 1.3464e-04 - val_loss: 1.2082e-04
Epoch 8/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 39s 68ms/step - loss: 1.2414e-04 - val_loss: 1.1281e-04
Epoch 9/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 42s 70ms/step - loss: 1.1634e-04 - val_loss: 1.0476e-04
Epoch 10/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 41s 71ms/step - loss: 1.0951e-04 - val_loss: 1.0771e-04
Epoch 11/30
453/453 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# 1. Generate reconstructions for test data
reconstructed_test = conv_autoencoder.predict(X_test)

# 2. Compute Reconstruction Error (MAE per sample)
test_mae = np.mean(np.abs(X_test - reconstructed_test.squeeze()), axis=1)

# 3. Compare mean error for majority class vs. minority classes in the test set
normal_mask = (y_test == 0)
for c in range(5):
    class_mse = np.mean(test_mae[y_test == c])
    print(f"Class {c} Average Reconstruction Error: {class_mse:.6f}")
# print(f"Mean Error for Normal (Class 0): {np.mean(test_mae[normal_mask]):.4f}")
# print(f"Mean Error for Anomaly (Classes 1-4): {np.mean(test_mae[~normal_mask]):.4f}")

685/685 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
Class 0 Average Reconstruction Error: 0.004826
Class 1 Average Reconstruction Error: 0.005939
Class 2 Average Reconstruction Error: 0.005191
Class 3 Average Reconstruction Error: 0.003041
Class 4 Average Reconstruction Error: 0.007113


In [ ]:
# Reconstruct the test set
X_test_pred = conv_autoencoder.predict(X_test)

# Calculate MSE for each sample across all 187 features
mse_test = np.mean(np.square(X_test - X_test_pred.squeeze()), axis=1)

685/685 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


In [ ]:
# Create binary ground truth: 0 for Normal, 1 for Anomaly
y_test_binary = (y_test != 0).astype(int)

# Lower percentile from 95 to 85 or 90 to improve Anomaly Recall
threshold = np.percentile(mse_test[y_test == 0], 80)
y_pred_binary = (mse_test > threshold).astype(int)


print(f"Chosen Anomaly Threshold: {threshold:.6f}")

Chosen Anomaly Threshold: 0.000082


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("--- Classification Report ---")
print(classification_report(y_test_binary, y_pred_binary, target_names=["Normal (0)", "Anomaly (1-4)"]))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test_binary, y_pred_binary))

--- Classification Report ---
               precision    recall  f1-score   support

   Normal (0)       0.91      0.80      0.85     18118
Anomaly (1-4)       0.39      0.61      0.47      3774

     accuracy                           0.77     21892
    macro avg       0.65      0.70      0.66     21892
 weighted avg       0.82      0.77      0.79     21892

--- Confusion Matrix ---
[[14494  3624]
 [ 1474  2300]]


**RES one class classification**

In [ ]:
# import numpy as np
# import pandas as pd
# import tensorflow as tf
# from sklearn.metrics import classification_report, f1_score
# from tensorflow import keras
# from tensorflow.keras import layers

# # 1. Binary Labels & Reshape
# y_train_b = (y_train != 0).astype(int)
# y_test_b = (y_test != 0).astype(int)
# X_train_c = np.expand_dims(X_train, -1)
# X_test_c = np.expand_dims(X_test, -1)


# # 2. Residual 1D-CNN Block Architecture
# def residual_block(x, filters, kernel_size=5):
#     res = x
#     x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(
#         x
#     )
#     x = layers.BatchNormalization()(x)
#     x = layers.Conv1D(filters, kernel_size, padding="same")(x)
#     x = layers.BatchNormalization()(x)
#     # Match channels for shortcut addition if needed
#     if res.shape[-1] != filters:
#         res = layers.Conv1D(filters, 1, padding="same")(res)
#     x = layers.add([x, res])
#     x = layers.Activation("relu")(x)
#     return layers.MaxPool1D(2)(x)


# inputs = layers.Input(shape=(187, 1))
# x = residual_block(inputs, 32)
# x = residual_block(x, 64)
# x = residual_block(x, 128)
# x = layers.Flatten()(x)
# x = layers.Dense(64, activation="relu")(x)
# x = layers.Dropout(0.4)(x)
# outputs = layers.Dense(1, activation="sigmoid")(x)

# model = keras.Model(inputs, outputs)

# # 3. Binary Crossentropy with Focal Loss effect via threshold search
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-3),
#     loss="binary_crossentropy",
#     metrics=["accuracy"],
# )

# # 4. Train with EarlyStopping
# early_stop = keras.callbacks.EarlyStopping(
#     monitor="val_loss", patience=5, restore_best_weights=True
# )
# model.fit(
#     X_train_c,
#     y_train_b,
#     epochs=25,
#     batch_size=128,
#     validation_split=0.15,
#     callbacks=[early_stop],
# )

# # 5. Predict Probabilities & Find Optimal Threshold for Max F1
# y_probs = model.predict(X_test_c).flatten()

# best_thresh = 0.5
# best_f1 = 0
# for t in np.arange(0.1, 0.6, 0.05):
#     f1 = f1_score(y_test_b, (y_probs > t).astype(int), pos_label=1)
#     if f1 > best_f1:
#         best_f1 = f1
#         best_thresh = t

# print(f"\nOptimal Decision Threshold: {best_thresh:.2f}")
# y_pred_opt = (y_probs > best_thresh).astype(int)
# print(
#     classification_report(
#         y_test_b, y_pred_opt, target_names=["Normal (0)", "Anomaly (1-4)"]
#     )
# )


import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Prepare Binary Labels (0 = Normal, 1 = Anomaly)
y_train_b = (y_train != 0).astype(int)
y_test_b = (y_test != 0).astype(int)

X_train_c = np.expand_dims(X_train, -1)
X_test_c = np.expand_dims(X_test, -1)

# Split train into train + validation (15%) to find threshold safely without leakage
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_c, y_train_b, test_size=0.15, random_state=42, stratify=y_train_b
)

# 2. Residual 1D-CNN Block
def residual_block(x, filters, kernel_size=5):
    res = x
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x)
    if res.shape[-1] != filters:
        res = layers.Conv1D(filters, 1, padding="same")(res)
    x = layers.add([x, res])
    x = layers.Activation("relu")(x)
    return layers.MaxPool1D(2)(x)

inputs = layers.Input(shape=(187, 1))
x = residual_block(inputs, 32)
x = residual_block(x, 64)
x = residual_block(x, 128)
x = layers.Flatten()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

stage1_model = keras.Model(inputs, outputs)

# 3. Use Binary Focal Crossentropy Loss to handle the 5:1 imbalance natively
focal_loss = keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.75)

stage1_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=focal_loss,
    metrics=["accuracy"]
)

# 4. Train with EarlyStopping & ReduceLROnPlateau
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]

stage1_model.fit(
    X_tr, y_tr,
    epochs=25,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

# 5. Determine Optimal Decision Cutoff on Validation Data ONLY
val_probs = stage1_model.predict(X_val).flatten()

best_thresh = 0.5
best_val_f1 = 0.0

for t in np.arange(0.1, 0.9, 0.02):
    f1 = f1_score(y_val, (val_probs > t).astype(int), pos_label=1)
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_thresh = t

print(f"\nOptimal Validation Decision Threshold: {best_thresh:.2f}")

# 6. Evaluate Cleanly on Unseen Test Data
test_probs = stage1_model.predict(X_test_c).flatten()
y_pred_opt = (test_probs > best_thresh).astype(int)

print("\n--- STAGE 1 FINAL EVALUATION ---")
print(classification_report(y_test_b, y_pred_opt, target_names=["Normal (0)", "Anomaly (1-4)"]))

Epoch 1/25
582/582 ━━━━━━━━━━━━━━━━━━━━ 162s 268ms/step - accuracy: 0.9512 - loss: 0.0440 - val_accuracy: 0.9603 - val_loss: 0.0755 - learning_rate: 0.0010
Epoch 2/25
582/582 ━━━━━━━━━━━━━━━━━━━━ 205s 274ms/step - accuracy: 0.9735 - loss: 0.0231 - val_accuracy: 0.9741 - val_loss: 0.0203 - learning_rate: 0.0010
Epoch 3/25
582/582 ━━━━━━━━━━━━━━━━━━━━ 158s 272ms/step - accuracy: 0.9788 - loss: 0.0185 - val_accuracy: 0.9797 - val_loss: 0.0176 - learning_rate: 0.0010
Epoch 4/25
582/582 ━━━━━━━━━━━━━━━━━━━━ 157s 270ms/step - accuracy: 0.9819 - loss: 0.0159 - val_accuracy: 0.9790 - val_loss: 0.0156 - learning_rate: 0.0010
Epoch 5/25
582/582 ━━━━━━━━━━━━━━━━━━━━ 158s 271ms/step - accuracy: 0.9834 - loss: 0.0144 - val_accuracy: 0.9836 - val_loss: 0.0150 - learning_rate: 0.0010
Epoch 6/25
582/582 ━━━━━━━━━━━━━━━━━━━━ 201s 270ms/step - accuracy: 0.9850 - loss: 0.0128 - val_accuracy: 0.9839 - val_loss: 0.0132 - learning_rate: 0.0010
Epoch 7/25
582/582 ━━━━━━━━━━━━━━━━━━━━ 208s 280ms/step - accura

**SECOND model based on RES outputs**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ==========================================
# 1. PREPARE DATA FOR STAGE 2 (ONLY ANOMALIES)
# ==========================================
# Filter out Class 0 from training & testing
train_anom_mask = y_train != 0
test_anom_mask = y_test != 0

X_train_s2 = X_train_c[train_anom_mask]
# Shift labels from [1, 2, 3, 4] -> [0, 1, 2, 3] for categorical training
y_train_s2 = (y_train[train_anom_mask] - 1).astype(int)

X_test_s2 = X_test_c[test_anom_mask]
y_test_s2 = (y_test[test_anom_mask] - 1).astype(int)

# Compute class weights for the 4 anomaly classes (handles small classes like Class 3)
class_weights_s2 = compute_class_weight(
    "balanced", classes=np.unique(y_train_s2), y=y_train_s2
)
cw_dict_s2 = dict(enumerate(class_weights_s2))


# ==========================================
# 2. BUILD STAGE 2 MULTI-CLASS RESNET MODEL
# ==========================================
def residual_block(x, filters, kernel_size=5):
  res = x
  x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
  x = layers.BatchNormalization()(x)
  x = layers.Conv1D(filters, kernel_size, padding="same")(x)
  x = layers.BatchNormalization()(x)
  if res.shape[-1] != filters:
    res = layers.Conv1D(filters, 1, padding="same")(res)
  x = layers.add([x, res])
  x = layers.Activation("relu")(x)
  return layers.MaxPool1D(2)(x)


inputs_s2 = layers.Input(shape=(187, 1))
x = residual_block(inputs_s2, 32)
x = residual_block(x, 64)
x = residual_block(x, 128)
x = layers.Flatten()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)
# Softmax output for 4 distinct anomaly classes
outputs_s2 = layers.Dense(4, activation="softmax")(x)

stage2_model = keras.Model(inputs_s2, outputs_s2)

stage2_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop_s2 = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

stage2_model.fit(
    X_train_s2,
    y_train_s2,
    epochs=30,
    batch_size=64,
    class_weight=cw_dict_s2,
    validation_split=0.15,
    callbacks=[early_stop_s2],
)

# Evaluate Stage 2 in isolation
y_pred_s2 = np.argmax(stage2_model.predict(X_test_s2), axis=1)
print("\n--- STAGE 2 ISOLATED EVALUATION ---")
print(
    classification_report(
        y_test_s2,
        y_pred_s2,
        target_names=["Class 1", "Class 2", "Class 3", "Class 4"],
    )
)


# ==========================================
# 3. END-TO-END TWO-STAGE INFERENCE PIPELINE
# ==========================================
def predict_full_pipeline(X_data, model_stage1, model_stage2, threshold_stage1):
  # Step A: Stage 1 Binary Screening
  probs_s1 = model_stage1.predict(X_data).flatten()
  is_anomaly = (probs_s1 > threshold_stage1).astype(int)

  # Initialize predictions array as Class 0 (Normal)
  final_predictions = np.zeros(len(X_data), dtype=int)

  # Step B: Route identified anomalies to Stage 2
  anomaly_indices = np.where(is_anomaly == 1)[0]
  if len(anomaly_indices) > 0:
    s2_probs = model_stage2.predict(X_data[anomaly_indices])
    # Map back from [0, 1, 2, 3] -> [1, 2, 3, 4]
    s2_classes = np.argmax(s2_probs, axis=1) + 1
    final_predictions[anomaly_indices] = s2_classes

  return final_predictions


# Evaluate Full Pipeline on Test Set
# final_test_preds = predict_full_pipeline(
#     X_test_c, model, stage2_model, best_thresh
# )
# With this corrected line:
final_test_preds = predict_full_pipeline(
    X_test_c, stage1_model, stage2_model, best_thresh)
print("\n--- COMPLETE TWO-STAGE PIPELINE EVALUATION ---")
print(
    classification_report(
        y_test.astype(int),
        final_test_preds,
        target_names=["Normal (0)", "Class 1", "Class 2", "Class 3", "Class 4"],
    )
)

Epoch 1/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 37s 155ms/step - accuracy: 0.8785 - loss: 0.3915 - val_accuracy: 0.0000e+00 - val_loss: 4.6657
Epoch 2/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 31s 152ms/step - accuracy: 0.9342 - loss: 0.2371 - val_accuracy: 4.4189e-04 - val_loss: 3.3094
Epoch 3/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 30s 151ms/step - accuracy: 0.9530 - loss: 0.1789 - val_accuracy: 0.8900 - val_loss: 0.3052
Epoch 4/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 31s 155ms/step - accuracy: 0.9601 - loss: 0.1451 - val_accuracy: 0.9872 - val_loss: 0.0470
Epoch 5/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 39s 147ms/step - accuracy: 0.9598 - loss: 0.1325 - val_accuracy: 0.9907 - val_loss: 0.0351
Epoch 6/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 31s 155ms/step - accuracy: 0.9635 - loss: 0.1227 - val_accuracy: 0.9925 - val_loss: 0.0371
Epoch 7/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 31s 152ms/step - accuracy: 0.9676 - loss: 0.1021 - val_accuracy: 0.9960 - val_loss: 0.0194
Epoch 8/30
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 150ms/step - accuracy: 0.9725 -

**OC-SVM**

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.metrics import classification_report
# from sklearn.svm import OneClassSVM

# # 1. Prepare Data
# # Train ONLY on Normal Class (0.0)
# X_train_normal = X_train[y_train == 0]

# # Test set ground truth: 0 for Normal, 1 for Anomaly
# y_test_binary = (y_test != 0).astype(int)

# # 2. Train One-Class SVM on Normal Data
# # nu is the expected outlier fraction (e.g., 5% to 10%)
# oc_svm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05)
# oc_svm.fit(X_train_normal)

# # 3. Predict on Test Set
# # OC-SVM returns +1 for Normal (inliers) and -1 for Anomalies (outliers)
# raw_preds = oc_svm.predict(X_test)
# y_pred_binary = np.where(raw_preds == -1, 1, 0)

# # 4. Evaluate
# print(
#     classification_report(
#         y_test_binary, y_pred_binary, target_names=["Normal (0)", "Anomaly (1-4)"]
#     )
# )

**Latent Space One-Class Classifier (Deep SVDD)**

In [ ]:
# from tensorflow.keras import models

# # 1. Extract the Encoder part of your trained conv_autoencoder
# # (Assumes layers up to the bottleneck/encoded layer)
# encoder = models.Model(
#     inputs=conv_autoencoder.input,
#     outputs=conv_autoencoder.get_layer("max_pooling1d_1").output,
# )

# # 2. Get latent features for Normal training data
# latent_normal = encoder.predict(np.expand_dims(X_train_major, -1))
# latent_normal_flat = latent_normal.reshape(latent_normal.shape[0], -1)

# # Compute Center Vector (Mean of normal representations)
# center = np.mean(latent_normal_flat, axis=0)

# # 3. Calculate Euclidean Distance to Center for Test Data
# latent_test = encoder.predict(np.expand_dims(X_test, -1))
# latent_test_flat = latent_test.reshape(latent_test.shape[0], -1)

# distances = np.linalg.norm(latent_test_flat - center, axis=1)

# # 4. Thresholding based on Distance (e.g., 90th percentile of normal training distance)
# normal_distances = np.linalg.norm(latent_normal_flat - center, axis=1)
# thresh = np.percentile(normal_distances, 90)

# y_pred_oneclass = (distances > thresh).astype(int)
# print(
#     classification_report(
#         (y_test != 0).astype(int),
#         y_pred_oneclass,
#         target_names=["Normal (0)", "Anomaly (1-4)"],
#     )
# )